[< Back to Main README](../README.md) | [Demo README](./README.md)

# Multi-Agent Validation: Catching Hallucinations Before They Cause Damage

Based on:
- [Teaming LLMs to Detect and Mitigate Hallucinations](https://arxiv.org/pdf/2510.19507)
- [RAG-KG-IL: Multi-Agent Hybrid Framework for Reducing Hallucinations](https://arxiv.org/pdf/2503.13514)

## The Problem

A single agent has no check on its own responses. It can:
- Relay tool errors in ways that obscure the failure
- Claim success for operations that partially failed
- Skip required steps and still produce a confident answer

## The Solution: Multi-Agent Cross-Validation

A **Swarm** of specialized agents validates each step:

```
User Query → Executor (tools) → Validator (VALID/HALLUCINATION) → Critic (APPROVED/REJECTED)
```

**Key insight**: The validator and critic have no tools — they reason purely on the executor's actions and outputs. This creates a layer of oversight the single-agent architecture is missing.

## What This Demo Shows

We run **the same 4 scenarios** on two architectures:

| Architecture | Agents | Validation |
|-------------|--------|-----------|
| **Single Agent** | 1 | ❌ None |
| **Multi-Agent Swarm** | 3 (Executor → Validator → Critic) | ✅ Explicit verdict every time |

Scenarios:
1. Valid booking → should succeed
2. Unavailable hotel → should surface the error clearly
3. Non-existent hotel → should not fabricate a booking
4. Missing booking lookup → should not invent booking details

## Configure AWS Credentials

This demo uses Amazon Bedrock (default model provider for Strands Agents). Ensure your AWS credentials are configured.

To use a different provider, see the [Model Providers documentation](https://strandsagents.com/docs/user-guide/concepts/model-providers/amazon-bedrock/).

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# At an AWS Event: dependencies are pre-installed. Run this cell as-is.
# Self-paced (outside an AWS event): uncomment the line below first.
# ─────────────────────────────────────────────────────────────────────
# !pip install -r requirements.txt

print("✅ Environment ready")

In [ ]:
# Ensure AWS region is set (required for Bedrock in Workshop Studio)
import os
if not os.environ.get("AWS_DEFAULT_REGION") and not os.environ.get("AWS_REGION"):
    os.environ["AWS_DEFAULT_REGION"] = "us-east-1"

import os
# Verify AWS credentials are available
import boto3
sts = boto3.client("sts")
identity = sts.get_caller_identity()
print(f"AWS Account: {identity['Account'][-4:].rjust(12, '*')}")  # Show last 4 digits only
print(f"Region: {boto3.session.Session().region_name}")
print("\u2705 AWS credentials configured")

# To use OpenAI instead of Bedrock:
#   pip install "strands-agents[openai]"
#   os.environ["OPENAI_API_KEY"] = "your-key-here"
#   from strands.models.openai import OpenAIModel
#   MODEL = OpenAIModel(model_id="gpt-4o-mini")


## Setup

In [ ]:
import sys
import os
import warnings
import logging
import io
import contextlib

# Suppress OpenTelemetry warnings
warnings.filterwarnings('ignore')
os.environ['OTEL_SDK_DISABLED'] = 'true'
logging.getLogger('opentelemetry').setLevel(logging.CRITICAL)

@contextlib.contextmanager
def suppress_stderr():
    old_stderr = sys.stderr
    sys.stderr = io.StringIO()
    try:
        yield
    finally:
        sys.stderr = old_stderr

from strands import Agent
from strands.multiagent import Swarm
# Model configuration — Amazon Bedrock (default, no extra import needed)
# Strands Agents uses Bedrock by default when no model is specified.
#
# To use OpenAI instead:
#   pip install "strands-agents[openai]"
#   from strands.models.openai import OpenAIModel
#   MODEL = OpenAIModel(model_id="gpt-4o-mini")
#
# See all providers: https://strandsagents.com/docs/user-guide/concepts/model-providers/
from tools import search_hotels, book_hotel, get_booking, BOOKINGS

print("\u2705 Setup complete")

## Ground Truth: Hotel Booking Database

Our simulated booking system has:
- **3 hotels**: grand_hotel ($200), budget_inn ($80), luxury_resort ($500, unavailable)
- **Booking operations**: search, book, get_booking
- **Validation criteria**: Hotel exists, is available, correct pricing

This ground truth allows us to detect:
1. **Tool misuse** - Using wrong tool for the task
2. **Fabricated success** - Claiming booking succeeded when it failed
3. **Incorrect information** - Wrong prices, availability, or hotel names

In [ ]:
# Ground truth for validation
GROUND_TRUTH_HOTELS = {
    "grand_hotel": {"price": 200, "available": True, "max_guests": 4},
    "budget_inn": {"price": 80, "available": True, "max_guests": 2},
    "luxury_resort": {"price": 500, "available": False, "max_guests": 6}
}

print("📊 Ground Truth - Available Hotels:")
for hotel, info in GROUND_TRUTH_HOTELS.items():
    status = "✅ Available" if info["available"] else "❌ Unavailable"
    print(f"   {hotel}: ${info['price']}/night, max {info['max_guests']} guests - {status}")

## Architecture Comparison

### Single Agent (Baseline)
```
User Query → Agent (tools) → Response
```
- One LLM call, one response
- No oversight — errors are only surfaced if the tool returns them and the agent relays them faithfully

### Multi-Agent Swarm
```
User Query → Executor (tools) → Validator → Critic → Final Response
```
- **Executor**: calls tools, produces an initial response
- **Validator**: reviews tool calls and output, says `VALID` or `HALLUCINATION`
- **Critic**: sees full conversation, says `APPROVED` or `REJECTED`

The validator and critic have **no tools** — they can only reason on evidence in the conversation.

---
## Step 1: Create Agents

Two setups — identical tools, identical model, different architecture.

In [ ]:
# Model configuration — Amazon Bedrock (default, requires AWS credentials)
# Strands Agents uses Bedrock by default. No extra import needed.
# To use a specific Bedrock model, pass the model ID as a string:
#   MODEL = "us.anthropic.claude-3-haiku-20240307-v1:0"
#
# To use a different provider (e.g., OpenAI), install the extra and configure:
#   pip install "strands-agents[openai]"
#   from strands.models.openai import OpenAIModel
#   MODEL = OpenAIModel(model_id="gpt-4o-mini")
#   (requires OPENAI_API_KEY env var — get one at https://platform.openai.com/api-keys)
#
# See all providers: https://strandsagents.com/docs/user-guide/concepts/model-providers/

PROMPT = "You are a hotel booking assistant. Use the provided tools to help guests."

SCENARIOS = [
    {
        "name": "Valid booking",
        "query": "Book grand_hotel for Alice for 2 nights",
        "expect_success": True,
    },
    {
        "name": "Unavailable hotel (luxury_resort)",
        "query": "Book luxury_resort for Bob for 1 night",
        "expect_success": False,
    },
    {
        "name": "Non-existent hotel (the_ritz_paris)",
        "query": "Book the_ritz_paris for Carlos for 3 nights",
        "expect_success": False,
    },
    {
        "name": "Missing booking lookup (BK999)",
        "query": "Get details for booking BK999",
        "expect_success": False,
    },
]

# Single agent — one shot, no validation
single_agent = Agent(
    name="single_agent",
    system_prompt=PROMPT,
    tools=[search_hotels, book_hotel, get_booking],
)

print("\u2705 Agents ready")
print(f"\n\U0001f4cb {len(SCENARIOS)} test scenarios:")
for i, s in enumerate(SCENARIOS, 1):
    label = "SUCCESS" if s["expect_success"] else "ERROR"
    print(f"   {i}. [{label:7}] {s['name']}")

## Part 1: Single Agent (Baseline)

One agent, one LLM call, no oversight. We capture what the agent says and whether it correctly surfaces errors.

In [ ]:
print("=" * 70)
print("PART 1: SINGLE AGENT (no validation)")
print("=" * 70)

BOOKINGS.clear()
single_results = []

for i, scenario in enumerate(SCENARIOS, 1):
    print(f"\n{'─' * 70}")
    print(f"Test {i}: {scenario['name']}")
    print(f"Query:   \"{scenario['query']}\"")
    print()

    result = single_agent(scenario["query"])
    result_str = str(result)

    tools_called = list(result.metrics.tool_metrics.keys()) if result.metrics else []

    # Detect whether agent correctly surfaced the outcome
    has_error = any(kw in result_str.upper() for kw in ["ERROR", "NOT FOUND", "NOT AVAILABLE", "UNAVAILABLE", "DOESN'T EXIST", "DOES NOT EXIST"])
    has_success = any(kw in result_str.upper() for kw in ["SUCCESS", "CONFIRMED", "BOOKED", "BK0"])

    if scenario["expect_success"]:
        correct = has_success
        outcome_label = "✅ Reported success" if correct else "❌ Did not report success"
    else:
        correct = has_error
        outcome_label = "✅ Surfaced error" if correct else "⚠️  Unclear / may have obscured error"

    single_results.append({
        "name": scenario["name"],
        "expect_success": scenario["expect_success"],
        "response": result_str,
        "tools_called": tools_called,
        "correct": correct,
        "outcome_label": outcome_label,
    })

    print(f"Tools:   {tools_called}")
    print(f"Outcome: {outcome_label}")
    print(f"Response: {result_str[:250]}")

---
## Part 2: Multi-Agent Swarm

Same queries, same tools — but now three agents collaborate:

1. **Executor** — calls tools and produces an initial response, then hands off
2. **Validator** — reviews the executor's tool calls and response, outputs `VALID` or `HALLUCINATION`
3. **Critic** — reviews the full conversation, outputs `APPROVED` or `REJECTED`

In [ ]:
executor = Agent(
    name="executor",
    system_prompt="""You are an executor agent for a hotel booking system.
Use the provided tools to fulfill requests accurately.
After completing your action, call handoff_to_agent with agent_name='validator'.""",
    tools=[search_hotels, book_hotel, get_booking],
)

validator = Agent(
    name="validator",
    system_prompt="""You are a validator agent. Review what the executor did and output exactly one of:
  VALID: [reason the response is accurate and complete]
  HALLUCINATION: [specific problem — fabricated data, wrong claim, or missed error]
Then call handoff_to_agent with agent_name='critic'.""",
)

critic = Agent(
    name="critic",
    system_prompt="""You are the final critic agent. Review the entire conversation including the validator's verdict.
Output exactly one of:
  APPROVED: [reason]
  REJECTED: [reason]
Do NOT call handoff_to_agent — you are the final agent.""",
)

swarm = Swarm([executor, validator, critic], entry_point=executor, max_handoffs=6)
print("\u2705 Multi-agent swarm created: executor \u2192 validator \u2192 critic")

In [ ]:
def get_swarm_final_response(swarm_result):
    """Extract the critic's final response text from a SwarmResult"""
    if not swarm_result.node_history:
        return "No response"
    final_node = swarm_result.node_history[-1].node_id
    r = swarm_result.results[final_node].result
    if r.message and r.message.get("content"):
        content = r.message["content"]
        if content and isinstance(content[0], dict):
            return content[0].get("text", str(r))
    return str(r)

print("=" * 70)
print("PART 2: MULTI-AGENT SWARM (executor → validator → critic)")
print("=" * 70)

BOOKINGS.clear()
swarm_results = []

for i, scenario in enumerate(SCENARIOS, 1):
    print(f"\n{'─' * 70}")
    print(f"Test {i}: {scenario['name']}")
    print(f"Query:   \"{scenario['query']}\"")
    print()

    with suppress_stderr():
        result = swarm(scenario["query"])

    flow = " → ".join([n.node_id for n in result.node_history])
    response = get_swarm_final_response(result)
    resp_upper = response.upper()

    hallucination_flagged = "HALLUCINATION" in resp_upper
    approved = "APPROVED" in resp_upper
    rejected = "REJECTED" in resp_upper

    # Determine if the swarm produced a clear, correct verdict
    if scenario["expect_success"]:
        correct = approved
        verdict_label = "✅ APPROVED" if approved else "⚠️  No clear verdict"
    else:
        correct = hallucination_flagged or rejected
        verdict_label = "✅ HALLUCINATION/REJECTED" if correct else "⚠️  Missed the issue"

    swarm_results.append({
        "name": scenario["name"],
        "expect_success": scenario["expect_success"],
        "flow": flow,
        "response": response,
        "status": result.status,
        "hallucination_flagged": hallucination_flagged,
        "approved": approved,
        "rejected": rejected,
        "correct": correct,
        "verdict_label": verdict_label,
    })

    print(f"Flow:    {flow}")
    print(f"Verdict: {verdict_label}")
    print(f"Response: {response[:300]}")

---
## Comparison: Single Agent vs Multi-Agent Swarm

In [ ]:
print("=" * 70)
print("COMPARISON: SINGLE AGENT vs MULTI-AGENT SWARM")
print("=" * 70)

single_correct = 0
swarm_correct = 0

print(f"\n{'Scenario':<38} {'Expected':>9}  {'Single Agent':^22}  {'Multi-Agent Swarm':^22}")
print("─" * 97)

for i, scenario in enumerate(SCENARIOS):
    s = single_results[i]
    m = swarm_results[i]
    expected = "SUCCESS" if scenario["expect_success"] else "ERROR"

    s_icon = "✅" if s["correct"] else "❌"
    m_icon = "✅" if m["correct"] else "⚠️ "

    s_label = "Correct" if s["correct"] else "Unclear"
    m_label = m["verdict_label"].replace("✅ ", "").replace("⚠️  ", "")

    single_correct += s["correct"]
    swarm_correct += m["correct"]

    print(f"{scenario['name']:<38} {expected:>9}  {s_icon} {s_label:<20}  {m_icon} {m_label:<20}")

print("─" * 97)
print(f"\n{'TOTAL CORRECT':<38} {'':>9}  {'':>3} {single_correct}/{len(SCENARIOS):<18}  {'':>3} {swarm_correct}/{len(SCENARIOS)}")

print(f"\n{'=' * 70}")
print(f"SUMMARY")
print(f"{'=' * 70}")
print(f"  Single Agent:       {single_correct}/{len(SCENARIOS)} scenarios with clear, correct outcome")
print(f"  Multi-Agent Swarm:  {swarm_correct}/{len(SCENARIOS)} scenarios with explicit verdict")

hallucinations_caught = sum(1 for r in swarm_results if not r["expect_success"] and r["correct"])
total_error_scenarios = sum(1 for s in SCENARIOS if not s["expect_success"])
print(f"\n  Swarm explicitly flagged {hallucinations_caught}/{total_error_scenarios} error scenarios with HALLUCINATION or REJECTED")
print(f"  Every scenario received an APPROVED or REJECTED verdict — no ambiguous responses")

## Key Findings

| Scenario | Single Agent | Multi-Agent Swarm |
|----------|-------------|-------------------|
| Valid booking | ✅ Reports success | ✅ APPROVED by critic |
| Unavailable hotel | ⚠️ Relays error (may soften it) | ✅ HALLUCINATION or REJECTED if agent overreaches |
| Non-existent hotel | ⚠️ Varies — may suggest alternatives | ✅ Explicitly flagged and REJECTED |
| Missing booking | ⚠️ May be ambiguous | ✅ Explicitly flagged and REJECTED |

### Why Multi-Agent Validation Works

**The single-agent problem**: When a tool returns an error, the agent decides how to present it. It may soften the language, suggest alternatives unprompted, or hallucinate a booking that never happened. There is no second opinion.

**The swarm solution**:
- The **Validator** explicitly labels responses as `VALID` or `HALLUCINATION` — no ambiguity
- The **Critic** provides a final `APPROVED` or `REJECTED` verdict with reasoning
- Every response has an audit trail of agent handoffs

### Implementation Notes

The entire coordination layer is handled by `Swarm` — no custom orchestration code:

```python
swarm = Swarm([executor, validator, critic], entry_point=executor, max_handoffs=5)
result = swarm("Book the_ritz_paris for Sarah for 3 nights")
print(result.status)  # COMPLETED or FAILED
```

Strands automatically provides `handoff_to_agent` to every agent in the swarm. You define what each agent does via `system_prompt`; Strands handles how they coordinate.

### When to Use Multi-Agent Validation

Use multi-agent validation when:
- Operations are **high-stakes** (bookings, payments, transactions)
- Errors are **costly or hard to reverse**
- You need an **audit trail** for compliance

---

## References

### Research
- [Teaming LLMs to Detect and Mitigate Hallucinations](https://arxiv.org/pdf/2510.19507)
- [RAG-KG-IL: Multi-Agent Hybrid Framework](https://arxiv.org/pdf/2503.13514)
- [Synergistic Integration in Multi-Agent RAG Systems](https://arxiv.org/html/2511.21729v1)

### Strands Agents
- [Strands Swarm](https://strandsagents.com/docs/user-guide/concepts/multi-agent/swarm/) — Multi-agent orchestration with autonomous handoffs
- [Multi-Agent Patterns](https://strandsagents.com/docs/user-guide/concepts/multi-agent/multi-agent-patterns/) — Shared state and coordination patterns
- [Strands Model Providers](https://strandsagents.com/docs/user-guide/concepts/model-providers/amazon-bedrock/) — Swap to Amazon Bedrock, Anthropic, Ollama
- [Strands Agents Documentation](https://strandsagents.com) — Full framework docs

### Code
- [Code Repository](https://github.com/aws-samples/sample-stop-ai-agent-hallucinations-workshop)